# Day 43：多 Agent 系统（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 6 周**：Agent 基础、工具调用与 LangGraph 状态机  
> **GitLab 工单**：`STARMIND-043` ｜ **开发分支**：`feat/day43-multi-agent-supervisor-team`  
> **配套 HTML5 课件**：`courseware/day43/index.html` ｜ **独立版本快照**：`versions/day43/`

---

## 🎙️ 老陈开场旁白

早啊各位！最近大家写单个 Agent 越写越顺手，但也一定发现了新瓶颈：当你把 20 个工具、3 份长达两千字的专业规范全都塞进同一个 System Prompt 时，大模型立刻开始“注意力稀释”，不是调错工具就是忘了输出格式。怎么办？软件工程里有微服务，人类社会有项目组——在 AI 里就是 **Multi-Agent（多 Agent 协同）**！今天老陈带大家对比 Supervisor、去中心化协作、层级团队三大经典架构，并亲手组建星辰数科的「行业研究三人组」：搜索员搜集情报、分析师算指标画结论、撰写员成文并接受主管质检打回。

### 承上启下
- **昨日基线**：昨天（Day 42）我们攻克了 LangGraph 的 Checkpointer 断点持久化、HITL 中断恢复、节点重试与并行章节写作 Agent，单个复杂 Agent 的工程底座已经非常扎实。
- **明日预告**：明天（Day 44）我们将把视野拓展到跨系统生态协议——Model Context Protocol（MCP），手写符合 JSON-RPC 2.0 规范的 MCP Server/Client，并构建 Agent 短期与长期双层记忆架构。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day43'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-043）

星辰数科战略发展部每周需要针对目标行业（如新能源、企业级 SaaS、工业视觉）输出一份包含竞品动态、财务估算与落地建议的深度研报。单 Agent 同时承担搜索、量化计算和公文排版时，经常出现数据引用错位或遗漏关键章节。

**今日迭代目标：**
- 对比 Supervisor（主管集中式）、Peer-to-Peer（去中心化交接）与 Hierarchical（层级树状）三种多 Agent 编排模式的适用边界。
- 实现通用的 `SupervisorOrchestrator` 状态机，支持共享黑板（Blackboard State）、按角色裁剪上下文视图与防踢皮球熔断。
- 落地星辰行业研究团队：`ResearcherAgent`（情报搜索员）+ `AnalystAgent`（数据分析师）+ `WriterAgent`（报告撰写员），支持主管质检打回重写。

**验收标准：**
- 给定行业研究主题后，Supervisor 能按「搜索 -> 分析 -> 撰写 -> 质检」顺序动态路由各专家节点。
- 当撰写员初稿缺少“风险提示”关键章节时，Supervisor 能将其打回重写并在下一轮通过验收。
- 设置 `max_rounds` 上限，模拟死循环路由时能触发安全熔断并返回降级报告，所有单测本地 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 什么时候真正需要 Multi-Agent？

不要一上来就把简单任务拆成 5 个 Agent！老陈总结的三条黄金判断标准：
1) **工具数量超过 8~10 个**，且分属不同业务域（如财务审批 vs 代码检索），单模型选择准确率开始抖动；
2) **需要角色对抗与交叉校验**（如“撰写员”与“法务审核员”互相独立，避免自己夸自己写得好）；
3) **安全权限隔离**：不同专家绑定不同的工具白名单，防止低权限问答链路越权触发写库操作。

> **⚠️ 老陈避坑**：坑点：把原本一个 Prompt 就能搞定的翻译+总结拆成 3 个 Agent，结果延迟翻了 3 倍、Token 费翻了 4 倍，效果反而因为信息传递损耗变差。

### 2. 2. 共享黑板（Blackboard）与上下文视图裁剪

在多 Agent 通信中，最忌讳把所有 Agent 的原始对话历史（包含几千字的网页抓取原文）原封不动广播给所有人。正确做法是维护一个结构化的 `TeamState` 黑板：每个专家只接收自己职责所需的 `context_view`（例如分析师只看 `facts`，撰写员只看 `analysis` 和 `review_feedback`），产出结构化工件后写回黑板。

> **⚠️ 老陈避坑**：坑点：如果允许所有专家随意覆写同一个 `messages` 字符串字段，很容易发生“后一个 Agent 覆盖前一个 Agent 关键证据”的脏写问题。

### 3. 3. LangGraph 中的真实 Multi-Agent 写法（Supervisor & Command）

在官方 LangGraph 中，主管路由通常利用结构化输出返回 `next: Literal['researcher', 'analyst', 'writer', 'FINISH']`，或者使用 `Command(goto='analyst', update={'facts': ...})` 在节点内部直接完成状态更新与路由跳转，代码结构与我们今天手写的 `SupervisorOrchestrator` 高度一致。

> **⚠️ 老陈避坑**：坑点：生产环境务必给 Supervisor 的路由输出加上白名单校验，防止大模型偶发幻觉输出一个不存在的 Agent 名字导致图引擎 `KeyError` 崩溃。


## 💻 实操步骤 1：步骤一：实现通用 Supervisor 多 Agent 路由状态机与权限隔离底座

- **目标文件**：`versions/day43/src/day43_multi_agent_patterns.py`
- **老陈代码旁白**：先搭底座！我们先写一个干净、可复用的 `SupervisorOrchestrator`：支持注册多个拥有独立 `allowed_tools` 的专家 Agent，通过 `RouterFunc` 动态决定下一跳，并内置非法路由拦截与 `max_rounds` 防死循环熔断。


In [ ]:
"""Day 43 - 步骤一：通用 Supervisor 多 Agent 路由状态机与上下文隔离底座。"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Protocol

FINISH = "FINISH"


class WorkerAgent(Protocol):
    """专家 Agent 统一协议：只读取裁剪后的上下文视图，返回结构化状态补丁。"""

    name: str
    role_desc: str
    allowed_tools: List[str]

    def run(self, state: Dict[str, Any], feedback: str = "") -> Dict[str, Any]:
        ...


@dataclass
class RouteDecision:
    next_agent: str
    reason: str
    feedback: str = ""


@dataclass
class MultiAgentResult:
    final_state: Dict[str, Any]
    routing_history: List[Dict[str, Any]] = field(default_factory=list)
    completed: bool = True
    truncated: bool = False
    rounds: int = 0


class SupervisorOrchestrator:
    """
    Supervisor 集中式多 Agent 编排器：
    - 维护共享黑板状态 (Blackboard State)
    - 每轮由 router_fn 决定派发给哪位专家或 FINISH
    - 内置路由白名单校验与 max_rounds 防踢皮球熔断
    """

    def __init__(
        self,
        router_fn: Callable[[Dict[str, Any], List[str]], RouteDecision],
        max_rounds: int = 8,
    ) -> None:
        self.router_fn = router_fn
        self.max_rounds = max_rounds
        self._workers: Dict[str, WorkerAgent] = {}

    def register_worker(self, worker: WorkerAgent) -> "SupervisorOrchestrator":
        if worker.name == FINISH:
            raise ValueError("专家名称不能与保留关键字 FINISH 冲突")
        self._workers[worker.name] = worker
        return self

    @property
    def worker_names(self) -> List[str]:
        return list(self._workers.keys())

    def invoke(self, initial_state: Dict[str, Any]) -> MultiAgentResult:
        state: Dict[str, Any] = dict(initial_state)
        history: List[Dict[str, Any]] = []

        for round_no in range(1, self.max_rounds + 1):
            decision = self.router_fn(state, self.worker_names)
            history.append(
                {
                    "round": round_no,
                    "next_agent": decision.next_agent,
                    "reason": decision.reason,
                    "feedback": decision.feedback,
                }
            )

            if decision.next_agent == FINISH:
                return MultiAgentResult(
                    final_state=state,
                    routing_history=history,
                    completed=True,
                    truncated=False,
                    rounds=round_no,
                )

            if decision.next_agent not in self._workers:
                raise ValueError(
                    f"Supervisor 路由目标非法: '{decision.next_agent}'，合法专家: {self.worker_names}"
                )

            worker = self._workers[decision.next_agent]
            patch = worker.run(state, feedback=decision.feedback)
            state.update(patch)

        # 超出最大轮次，触发防踢皮球安全熔断
        return MultiAgentResult(
            final_state=state,
            routing_history=history,
            completed=False,
            truncated=True,
            rounds=self.max_rounds,
        )


if __name__ == "__main__":
    class EchoWorker:
        def __init__(self, name: str, role_desc: str, allowed_tools: List[str]) -> None:
            self.name = name
            self.role_desc = role_desc
            self.allowed_tools = allowed_tools

        def run(self, state: Dict[str, Any], feedback: str = "") -> Dict[str, Any]:
            done = list(state.get("done", []))
            done.append(f"{self.name}:{feedback or 'ok'}")
            return {"done": done}

    def demo_router(state: Dict[str, Any], workers: List[str]) -> RouteDecision:
        done = state.get("done", [])
        if len(done) == 0:
            return RouteDecision(next_agent=workers[0], reason="先执行第一步采集")
        return RouteDecision(next_agent=FINISH, reason="采集已完成，结束流程")

    orch = SupervisorOrchestrator(router_fn=demo_router, max_rounds=4)
    orch.register_worker(EchoWorker("collector", "采集员", ["search"]))
    res = orch.invoke({"topic": "AI Agent 行业趋势"})
    print("完成状态:", res.completed, "| 轮次:", res.rounds, "| 轨迹:", res.routing_history)


### 🔍 步骤 1 代码深度解析

1. `WorkerAgent` 协议约束所有专家只需实现 `run(state, feedback) -> dict`，不直接修改全局状态，保证状态流转清晰可测。
2. `SupervisorOrchestrator.invoke` 每次先校验 `decision.next_agent` 是否在注册白名单内，防止模型幻觉输出未知角色。
3. 循环到达 `max_rounds` 后自动返回 `truncated=True`，为线上多 Agent 团队兜住成本和死循环底线。

**预期运行输出参考：**
```text
完成状态: True | 轮次: 2 | 轨迹: [{'round': 1, 'next_agent': 'collector', 'reason': '先执行第一步采集', 'feedback': ''}, {'round': 2, 'next_agent': 'FINISH', 'reason': '采集已完成，结束流程', 'feedback': ''}]
```


## 💻 实操步骤 2：步骤二：组建星辰行业研究团队（搜索员 + 分析师 + 撰写员 + 质检打回闭环）

- **目标文件**：`versions/day43/src/day43_research_team.py`
- **老陈代码旁白**：底座搭好了，现在我们正式把星辰数科的“行业研究三人组”装进 `SupervisorOrchestrator`！特别注意看撰写员第一次故意漏掉「风险提示」时，主管是如何精准质检打回、并由撰写员在第二轮定向补齐后顺利交付终稿的。


In [ ]:
"""Day 43 - 步骤二：星辰行业研究多 Agent 团队（搜索员 + 分析师 + 撰写员）。"""

from __future__ import annotations

from typing import Any, Dict, List

from src.day43_multi_agent_patterns import (
    FINISH,
    MultiAgentResult,
    RouteDecision,
    SupervisorOrchestrator,
)

# 本地模拟企业行业情报库（保证 100% 离线确定性运行）
MOCK_INDUSTRY_DB: Dict[str, List[Dict[str, Any]]] = {
    "企业级Agent": [
        {"source": "2025中国AI应用白皮书", "metric": "市场规模(亿元)", "y2024": 180.0, "y2025": 315.0},
        {"source": "星辰客户调研库", "metric": "头部企业落地率(%)", "y2024": 22.0, "y2025": 48.0},
        {"source": "运维成本报告", "metric": "平均人效提升(%)", "y2024": 18.0, "y2025": 36.0},
    ],
    "工业视觉": [
        {"source": "智能制造年鉴", "metric": "市场规模(亿元)", "y2024": 120.0, "y2025": 156.0},
        {"source": "产线质检基准", "metric": "缺陷漏检率下降(%)", "y2024": 25.0, "y2025": 40.0},
    ],
}


class ResearcherAgent:
    """搜索员：只拥有情报检索权限，负责收集带出处的结构化事实。"""

    name = "researcher"
    role_desc = "行业情报搜索员，负责检索权威数据与来源"
    allowed_tools = ["industry_search"]

    def run(self, state: Dict[str, Any], feedback: str = "") -> Dict[str, Any]:
        topic = state["topic"]
        records = MOCK_INDUSTRY_DB.get(
            topic,
            [{"source": "公开研报库", "metric": "行业指数", "y2024": 100.0, "y2025": 125.0}],
        )
        return {"facts": records, "fact_count": len(records)}


class AnalystAgent:
    """分析师：不触网，仅基于黑板上的 facts 进行同比增速与关键洞察测算。"""

    name = "analyst"
    role_desc = "量化数据分析师，负责计算同比增速并提炼核心趋势"
    allowed_tools = ["metric_calculator"]

    def run(self, state: Dict[str, Any], feedback: str = "") -> Dict[str, Any]:
        facts = state.get("facts", [])
        insights: List[Dict[str, Any]] = []
        for item in facts:
            base = item["y2024"]
            curr = item["y2025"]
            yoy_pct = round(((curr - base) / base) * 100.0, 2) if base else 0.0
            insights.append(
                {
                    "metric": item["metric"],
                    "source": item["source"],
                    "y2024": base,
                    "y2025": curr,
                    "yoy_growth_pct": yoy_pct,
                }
            )
        avg_growth = round(sum(x["yoy_growth_pct"] for x in insights) / max(len(insights), 1), 2)
        return {
            "analysis": {
                "metrics": insights,
                "avg_yoy_growth_pct": avg_growth,
                "verdict": "高速扩张期" if avg_growth >= 50.0 else "稳健增长期",
            }
        }


class WriterAgent:
    """撰写员：基于分析师结论撰写研报，并响应主管的质检打回意见定向修订。"""

    name = "writer"
    role_desc = "资深行业研报撰写员，负责结构化公文排版与修订"
    allowed_tools = ["markdown_formatter"]

    def run(self, state: Dict[str, Any], feedback: str = "") -> Dict[str, Any]:
        topic = state["topic"]
        analysis = state["analysis"]
        revisions = int(state.get("writer_revisions", 0)) + 1

        lines = [
            f"# 《{topic}》行业深度研究报告",
            "",
            f"## 一、核心结论（阶段判定：{analysis['verdict']}）",
            f"综合各核心维度测算，该赛道平均同比增速达到 **{analysis['avg_yoy_growth_pct']}%**。",
            "",
            "## 二、关键量化指标",
        ]
        for m in analysis["metrics"]:
            lines.append(
                f"- {m['metric']}：2024年 {m['y2024']} -> 2025年 {m['y2025']}（同比 +{m['yoy_growth_pct']}%，来源：{m['source']}）"
            )

        # 演示真实质检闭环：首版未包含风险提示，待主管打回并给出 feedback 后补齐
        if "风险提示" in feedback or revisions >= 2:
            lines.extend(
                [
                    "",
                    "## 三、风险提示与落地建议",
                    "- 需重点防范模型幻觉引发的合规风险，建议引入 Human-in-the-loop 审批护栏。",
                    "- 建议优先从高 ROI 内部提效场景切入，控制 Token 推理成本。",
                ]
            )

        return {"draft": "\n".join(lines), "writer_revisions": revisions}


def research_supervisor_router(state: Dict[str, Any], _workers: List[str]) -> RouteDecision:
    """主管路由决策函数：按 情报 -> 分析 -> 撰写 -> 质检验收 的状态推进。"""
    if not state.get("facts"):
        return RouteDecision(next_agent="researcher", reason="缺少行业原始情报，派发搜索员")
    if not state.get("analysis"):
        return RouteDecision(next_agent="analyst", reason="已有原始事实，派发分析师测算同比增速")
    draft = state.get("draft", "")
    if not draft:
        return RouteDecision(next_agent="writer", reason="量化分析完成，派发撰写员起草初稿")
    if "风险提示" not in draft:
        return RouteDecision(
            next_agent="writer",
            reason="质检未通过：研报缺少合规与风险提示章节，打回重写",
            feedback="请补充「风险提示」章节，说明模型幻觉与成本控制建议。",
        )
    return RouteDecision(next_agent=FINISH, reason="研报数据完整且通过风险提示质检，准予发布")


def build_research_team(max_rounds: int = 8) -> SupervisorOrchestrator:
    orch = SupervisorOrchestrator(router_fn=research_supervisor_router, max_rounds=max_rounds)
    orch.register_worker(ResearcherAgent())
    orch.register_worker(AnalystAgent())
    orch.register_worker(WriterAgent())
    return orch


if __name__ == "__main__":
    team = build_research_team()
    result: MultiAgentResult = team.invoke({"topic": "企业级Agent"})
    print(f"协作完成: {result.completed} | 总调度轮次: {result.rounds}")
    for step in result.routing_history:
        print(f"  Round {step['round']}: -> {step['next_agent']} ({step['reason']})")
    print("\n" + result.final_state["draft"])


### 🔍 步骤 2 代码深度解析

1. 三名专家各司其职且严格隔离：`ResearcherAgent` 只搜事实，`AnalystAgent` 只算增速，`WriterAgent` 只管成文。
2. `research_supervisor_router` 在发现初稿缺少 `风险提示` 后，自动附带 `feedback` 将任务打回给 `writer`，体现了多 Agent 独有的“生成-质检-修订”闭环优势。
3. 整个流程完全基于本地确定性数据运行，方便写入 CI 自动化回归测试。

**预期运行输出参考：**
```text
协作完成: True | 总调度轮次: 5
  Round 1: -> researcher (缺少行业原始情报，派发搜索员)
  Round 2: -> analyst (已有原始事实，派发分析师测算同比增速)
  Round 3: -> writer (量化分析完成，派发撰写员起草初稿)
  Round 4: -> writer (质检未通过：研报缺少合规与风险提示章节，打回重写)
  Round 5: -> FINISH (研报数据完整且通过风险提示质检，准予发布)

# 《企业级Agent》行业深度研究报告
...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day43/tests/test_day43.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 43 自动化单元测试：验证 Supervisor 路由状态机、研究团队质检打回与防死循环熔断。"""

import pytest
from src.day43_multi_agent_patterns import (
    FINISH,
    RouteDecision,
    SupervisorOrchestrator,
)
from src.day43_research_team import build_research_team


def test_research_team_full_collaboration_with_revision():
    team = build_research_team(max_rounds=8)
    res = team.invoke({"topic": "企业级Agent"})
    assert res.completed is True
    assert res.truncated is False
    assert res.rounds == 5
    routed_order = [item["next_agent"] for item in res.routing_history]
    assert routed_order == ["researcher", "analyst", "writer", "writer", FINISH]
    assert res.final_state["writer_revisions"] == 2
    assert "风险提示" in res.final_state["draft"]
    assert res.final_state["analysis"]["verdict"] == "高速扩张期"


def test_supervisor_truncates_on_ping_pong_loop():
    class DummyWorker:
        name = "loop_worker"
        role_desc = "测试死循环"
        allowed_tools = []

        def run(self, state, feedback=""):
            return {"count": state.get("count", 0) + 1}

    orch = SupervisorOrchestrator(
        router_fn=lambda _s, _w: RouteDecision(next_agent="loop_worker", reason="永远打回"),
        max_rounds=3,
    )
    orch.register_worker(DummyWorker())
    res = orch.invoke({})
    assert res.completed is False
    assert res.truncated is True
    assert res.rounds == 3
    assert res.final_state["count"] == 3


def test_supervisor_rejects_unknown_worker_route():
    orch = SupervisorOrchestrator(
        router_fn=lambda _s, _w: RouteDecision(next_agent="ghost_agent", reason="幻觉角色"),
        max_rounds=3,
    )
    with pytest.raises(ValueError, match="路由目标非法"):
        orch.invoke({})

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为研究团队新增法务合规审核员（ReviewerAgent）

**题目要求：**
编写一个 `ComplianceReviewerAgent`，扫描研报 `draft` 中是否包含“绝对第一”“稳赚不赔”等违规广告词，若命中则返回 `compliance_passed=False` 与违规词列表。

**思路提示：** 遍历违禁词列表检查 `word in draft`，并将命中的词收集到 `violations` 字段返回。


In [ ]:
"""Day 43 作业 1 参考答案：法务合规审核员 Agent。"""

from typing import Any, Dict, List


class ComplianceReviewerAgent:
    name = "compliance_reviewer"
    BANNED_WORDS = ["绝对第一", "稳赚不赔", "100%保本"]

    def run(self, state: Dict[str, Any], feedback: str = "") -> Dict[str, Any]:
        draft = state.get("draft", "")
        hits: List[str] = [w for w in self.BANNED_WORDS if w in draft]
        return {"compliance_passed": len(hits) == 0, "violations": hits}


if __name__ == "__main__":
    agent = ComplianceReviewerAgent()
    r1 = agent.run({"draft": "本方案行业绝对第一，建议采纳。"})
    r2 = agent.run({"draft": "本方案在基准测试中表现优异，附风险提示。"})
    print("违规检测:", r1, "| 合规检测:", r2)
    assert r1["compliance_passed"] is False and r1["violations"] == ["绝对第一"]
    assert r2["compliance_passed"] is True


**答案解析与评分标准：**

把合规校验抽离成独立的审核 Agent（或确定性规则节点），比在撰写员 Prompt 里写十遍“不要用违禁词”可靠得多。


### 作业 2（企业实战）：实现单角色最大返工次数限制器

**题目要求：**
统计路由历史中每个角色被连续调用的次数，编写 `detect_excessive_rework(history, limit=2)`，当同一角色连续返工超过 `limit` 次时返回 `True` 触发人工介入。

**思路提示：** 遍历 `routing_history` 中的 `next_agent`，维护当前连续计数 `streak`。


In [ ]:
"""Day 43 作业 2 参考答案：单角色连续返工熔断检测。"""

from typing import Any, Dict, List


def detect_excessive_rework(history: List[Dict[str, Any]], limit: int = 2) -> bool:
    last_agent = None
    streak = 0
    for item in history:
        agent = item["next_agent"]
        if agent == last_agent:
            streak += 1
        else:
            last_agent = agent
            streak = 1
        if streak > limit:
            return True
    return False


if __name__ == "__main__":
    ok_hist = [{"next_agent": "writer"}, {"next_agent": "writer"}]
    bad_hist = [{"next_agent": "writer"}, {"next_agent": "writer"}, {"next_agent": "writer"}]
    assert detect_excessive_rework(ok_hist, limit=2) is False
    assert detect_excessive_rework(bad_hist, limit=2) is True
    print("连续返工熔断检测通过")


**答案解析与评分标准：**

区分全局 `max_rounds` 与单角色连续返工上限，可以更早发现某位专家“卡壳”并及时降级或转人工。


### 作业 3（架构进阶）：按角色生成最小化上下文视图（Context Projection）

**题目要求：**
编写 `project_context(state, role)` 函数：只把 `researcher` 需要的 `topic`、`analyst` 需要的 `facts`、`writer` 需要的 `topic/analysis` 投影出来，避免无关大字段泄露或浪费 Token。

**思路提示：** 定义一个 `ROLE_FIELDS` 白名单映射字典，按角色过滤 `state` 键值。


In [ ]:
"""Day 43 作业 3 参考答案：多 Agent 最小化上下文视图投影。"""

from typing import Any, Dict, Set

ROLE_FIELDS: Dict[str, Set[str]] = {
    "researcher": {"topic"},
    "analyst": {"topic", "facts"},
    "writer": {"topic", "analysis", "writer_revisions"},
}


def project_context(state: Dict[str, Any], role: str) -> Dict[str, Any]:
    allowed = ROLE_FIELDS.get(role, set())
    return {k: v for k, v in state.items() if k in allowed}


if __name__ == "__main__":
    full_state = {"topic": "企业级Agent", "facts": [1, 2], "raw_html": "<html>超长噪音</html>", "analysis": {"ok": True}}
    w_view = project_context(full_state, "writer")
    print("Writer 视图:", w_view)
    assert "raw_html" not in w_view and "facts" not in w_view and "analysis" in w_view


**答案解析与评分标准：**

上下文投影（Context Projection）是多 Agent 生产省钱提质的秘密武器，通常可减少 40%~60% 的冗余输入 Token。


---

## 🎯 老陈收尾复盘

今天这一课含金量极高！大家不仅跑通了 Supervisor 调度三名专家 Agent 的完整闭环，还亲手解决了多 Agent 生产落地最头疼的“互相踢皮球死循环”和“上下文爆炸”问题。记住：不要为了炫技拆分 Agent，只有当角色职责、工具权限或 Prompt 专长明显隔离时，Multi-Agent 才会带来正收益。明天我们进军 MCP 协议与长短期记忆！
